# 01 — Data Preparation: ImageNet-1M (ZJU)

This notebook **downloads, verifies and explores** the ImageNet-1M benchmark used in all
subsequent experiments.

| Split | Vectors | Dim | Size |
|---|---|---|---|
| Base | 1 281 167 | 2048 | ≈ 9.4 GB |
| Query | 25 000 | 2048 | ≈ 195 MB |
| Ground truth | 25 000 × 100 nn | int32 | ≈ 9.6 MB |

Source: <http://www.cad.zju.edu.cn/home/dengcai/Data/ANNS/ANNSData.html>

Sections:
1. Download & integrity check
2. Dataset summary
3. L2-norm distribution
4. Per-dimension statistics
5. 2-D PCA scatter
6. Ground-truth verification against `IndexFlatL2`
7. Persist a small HDF5 with query + GT for downstream notebooks

In [ ]:
import os, sys, time, subprocess, hashlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import h5py
import faiss
import psutil
from tqdm import tqdm

sys.path.insert(0, str(Path.cwd()))
import utils

DATA = Path('data')
DATA.mkdir(exist_ok=True)
LAB_LIGHT = int(os.environ.get('LAB_LIGHT', '0'))
OUT_RUN = utils.run_mode()
DOCS_IMG = utils.plots_dir()
print(f'OUT_RUN={OUT_RUN}  plots → {DOCS_IMG}')

PATHS = {
    'base':  DATA / 'imagenet_base.fvecs',
    'query': DATA / 'imagenet_query.fvecs',
    'gt':    DATA / 'imagenet_groundtruth.ivecs',
    'h5':    DATA / 'imagenet1m.h5',
}

URLS = {
    'base.tar':  'http://www.cad.zju.edu.cn/home/dengcai/Data/ANNS/imagenet_base.fvecs.tar.gz',
    'query.tar': 'http://www.cad.zju.edu.cn/home/dengcai/Data/ANNS/imagenet_query.fvecs.tar.gz',
    'gt.tar':    'http://www.cad.zju.edu.cn/home/dengcai/Data/ANNS/imagenet_groundtruth.ivecs.tar.gz',
}

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
print('faiss', faiss.__version__, '| threads', faiss.omp_get_max_threads())
print(f"RAM total {psutil.virtual_memory().total/1e9:.1f} GB | free {psutil.virtual_memory().available/1e9:.1f} GB")

## 1 · Download & extract

The cell below is **idempotent** — it does nothing if the `.fvecs` / `.ivecs` files already
exist. The 9.4 GB base file is the slow one; expect 30–90 min on a 3-5 MB/s link.

In [ ]:
def _curl(url: str, out: Path) -> None:
    print(f"  → {url}")
    subprocess.check_call(['curl', '-fSL', '--retry', '5', '-o', str(out), url])

def _extract(tar: Path, expected: Path) -> None:
    if expected.exists():
        return
    print(f"  ✱ tar xzf {tar.name}")
    subprocess.check_call(['tar', 'xzf', str(tar), '-C', str(tar.parent)])

for name, fvecs in [('base', PATHS['base']), ('query', PATHS['query']), ('gt', PATHS['gt'])]:
    if fvecs.exists():
        sz = fvecs.stat().st_size / 1e9
        print(f"✓ {fvecs.name}  ({sz:.2f} GB) already present")
        continue
    tar = fvecs.parent / (fvecs.stem + ('.fvecs.tar.gz' if fvecs.suffix == '.fvecs' else '.ivecs.tar.gz'))
    if not tar.exists():
        _curl(URLS[f'{name}.tar'], tar)
    _extract(tar, fvecs)
    print(f"✓ {fvecs.name}  ({fvecs.stat().st_size/1e9:.2f} GB)")

## 2 · Dataset summary

In [ ]:
n_base, dim = utils._count_vecs(str(PATHS['base']), dtype_bytes=4)
n_query, dim_q = utils._count_vecs(str(PATHS['query']), dtype_bytes=4)
n_gt, k_gt = utils._count_vecs(str(PATHS['gt']), dtype_bytes=4)
summary = pd.DataFrame({
    'split': ['base', 'query', 'groundtruth'],
    'n':     [n_base, n_query, n_gt],
    'cols':  [dim, dim_q, k_gt],
    'bytes': [PATHS['base'].stat().st_size, PATHS['query'].stat().st_size, PATHS['gt'].stat().st_size],
})
summary['MB'] = (summary.bytes / 1024 / 1024).round(1)
display(summary)
assert dim == dim_q == 2048, "expected 2048-D vectors"
assert n_query == n_gt == 25_000
print(f"base vectors: {n_base:,}  ·  query/gt: {n_query:,}  ·  dim: {dim}")

## 3 · L2-norm distribution

These features are L2-normalised-ish ResNet-style embeddings.  Plotting the histogram of
norms reveals whether the dataset is normalised (so cosine ≡ L2) or arbitrary scale.

In [ ]:
SAMPLE_N = 100_000
rng = np.random.default_rng(42)
sample_ids = np.sort(rng.choice(n_base, SAMPLE_N, replace=False))

# Read sample by memmap to avoid loading 10 GB
mm = utils.read_fvecs(str(PATHS['base']), mmap=True)
sample = np.array(mm[sample_ids], dtype=np.float32)  # 100k × 2048 ≈ 780 MB
print('sample shape', sample.shape, 'RAM', sample.nbytes/1e6, 'MB')

norms = np.linalg.norm(sample, axis=1)
print(f"norm min={norms.min():.3f}  median={np.median(norms):.3f}  max={norms.max():.3f}  std={norms.std():.3f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(norms, bins=80, color='steelblue', edgecolor='white')
ax[0].set_title('L2 norms (100k sample)')
ax[0].set_xlabel('||v||₂'); ax[0].set_ylabel('count')

q_norms = np.linalg.norm(utils.read_fvecs(str(PATHS['query'])), axis=1)
ax[1].hist(q_norms, bins=80, color='darkorange', edgecolor='white')
ax[1].set_title('L2 norms (25k queries)')
ax[1].set_xlabel('||v||₂'); ax[1].set_ylabel('count')
plt.tight_layout(); plt.savefig(DOCS_IMG / '01_norms.png', dpi=120, bbox_inches='tight'); plt.show()

## 4 · Per-dimension mean / std

If any subset of dimensions has very different scale, IVF k-means and HNSW edges can be
dominated by those axes.  This plot is a sanity check that the embedding is reasonably
isotropic.

In [ ]:
mu = sample.mean(axis=0)
sd = sample.std(axis=0)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(mu, lw=0.4, color='steelblue')
ax[0].axhline(mu.mean(), color='k', ls='--', lw=0.7, label=f'global μ={mu.mean():.3f}')
ax[0].set_title('Per-dimension mean'); ax[0].set_xlabel('dim'); ax[0].legend()
ax[1].plot(sd, lw=0.4, color='crimson')
ax[1].axhline(sd.mean(), color='k', ls='--', lw=0.7, label=f'global σ̄={sd.mean():.3f}')
ax[1].set_title('Per-dimension std'); ax[1].set_xlabel('dim'); ax[1].legend()
plt.tight_layout(); plt.savefig(DOCS_IMG / '01_dim_stats.png', dpi=120, bbox_inches='tight'); plt.show()

print(f"min std {sd.min():.3f}  max std {sd.max():.3f}  ratio {sd.max()/sd.min():.1f}×")

## 5 · 2-D PCA scatter (10k sample)

A cheap 2-D PCA gives a feel for how the 1 000 ImageNet classes are arranged in this
embedding.  We colour 10 random clusters from a small `KMeans` on top of the PCA to make
structure visible — this is just a visualisation, no labels are used.

In [ ]:
small_n = 10_000
small = sample[:small_n]

# Quick centred SVD-based PCA, top-2 components
small_c = small - small.mean(axis=0)
U, S, Vt = np.linalg.svd(small_c, full_matrices=False)
proj = small_c @ Vt[:2].T

# 10-way k-means on the embedding (fast with faiss)
km = faiss.Kmeans(d=dim, k=10, niter=20, seed=1, verbose=False)
km.train(small)
_, labels = km.index.search(small, 1)
labels = labels.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(proj[:, 0], proj[:, 1], c=labels, cmap='tab10', s=4, alpha=0.6)
ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
ax.set_title('Top-2 PCA of ImageNet-1M embeddings (10k sample, k-means k=10 colours)')
plt.colorbar(sc, ax=ax, label='cluster id')
plt.tight_layout(); plt.savefig(DOCS_IMG / '01_pca.png', dpi=120, bbox_inches='tight'); plt.show()

print(f"PC1 explains {S[0]**2 / (S**2).sum() * 100:.2f} %, "
      f"PC2 explains {S[1]**2 / (S**2).sum() * 100:.2f} %")
del small, small_c, U, S, Vt, sample; import gc; gc.collect()

## 6 · Ground-truth verification

**Full run (default):** `IndexFlatL2` over the **entire base** (streamed from memmap) on
5 000 queries; Recall@100 vs the published GT must be ≈ 1.0.

**Light run (`LAB_LIGHT=1`):** Flat index on the first **100 000** base vectors only,
300 queries, plus a **numpy cross-check** on query 0 (exact top-100 by brute force on that
subset).  This avoids the multi-hour full-base pass while still validating readers + FAISS.

In [ ]:
# Load query + supplied GT
queries = utils.read_fvecs(str(PATHS['query']))   # 25k × 2048
gt = utils.read_ivecs(str(PATHS['gt']))           # 25k × 100
print('queries', queries.shape, 'gt', gt.shape)

LAB_LIGHT = int(os.environ.get('LAB_LIGHT', '0'))
mm = utils.read_fvecs(str(PATHS['base']), mmap=True)

if LAB_LIGHT:
    SUB_N, SAMPLE_Q = 100_000, 300
    qs = queries[:SAMPLE_Q]
    gts = gt[:SAMPLE_Q]
    xb_sub = np.ascontiguousarray(mm[:SUB_N])
    print(f'LAB_LIGHT=1 — Flat on first {SUB_N:,} base vectors, {SAMPLE_Q} queries')
    flat = faiss.IndexFlatL2(dim)
    t0 = time.perf_counter()
    flat.add(xb_sub)
    print(f'  add {flat.ntotal:,} in {time.perf_counter()-t0:.1f}s')
    t0 = time.perf_counter()
    D_flat, I_flat = flat.search(qs, 100)
    print(f'  search {time.perf_counter()-t0:.2f}s')

    # numpy exact top-100 for query 0 on the same subset
    d0 = np.sum((xb_sub - qs[0]) ** 2, axis=1)
    true_idx = np.argpartition(d0, 99)[:100]
    true_idx = true_idx[np.argsort(d0[true_idx])]
    faiss_idx = np.sort(I_flat[0])
    np_idx = np.sort(true_idx)
    assert np.array_equal(faiss_idx, np_idx), 'FAISS Flat vs numpy mismatch on subset'
    print('  ✓ numpy cross-check on query 0 passed')

    r1   = utils.compute_recall(I_flat, gts, 1)
    r10  = utils.compute_recall(I_flat, gts, 10)
    r100 = utils.compute_recall(I_flat, gts, 100)
    print(f'Flat(subset) vs *full-base* supplied GT — R@1 {r1:.4f}  R@10 {r10:.4f}  R@100 {r100:.4f}')
    print('  (Low recall here is expected: GT neighbours often lie outside the first 100k IDs.)')
else:
    SAMPLE_Q = 5_000
    qs = queries[:SAMPLE_Q]
    gts = gt[:SAMPLE_Q]
    print('Building IndexFlatL2 on full base (memmap stream-add)...')
    flat = faiss.IndexFlatL2(dim)
    BATCH = 50_000
    t0 = time.perf_counter()
    for i in tqdm(range(0, n_base, BATCH)):
        flat.add(np.ascontiguousarray(mm[i:i+BATCH]))
    print(f'  added {flat.ntotal:,} in {time.perf_counter()-t0:.1f}s  | RSS {psutil.Process().memory_info().rss/1e9:.2f} GB')
    print('Searching 5k queries × k=100...')
    t0 = time.perf_counter()
    D_flat, I_flat = flat.search(qs, 100)
    print(f'  search {time.perf_counter()-t0:.1f}s  ({SAMPLE_Q/(time.perf_counter()-t0):.1f} qps)')
    r1   = utils.compute_recall(I_flat, gts, 1)
    r10  = utils.compute_recall(I_flat, gts, 10)
    r100 = utils.compute_recall(I_flat, gts, 100)
    print(f'Flat vs supplied GT — R@1 {r1:.4f}  R@10 {r10:.4f}  R@100 {r100:.4f}')

In [ ]:
# Per-query recall@100 distribution (full run only asserts against GT)
per_q = np.array([np.intersect1d(I_flat[i], gts[i]).size / 100 for i in range(SAMPLE_Q)])
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 21) if per_q.max() > per_q.min() + 1e-6 else np.linspace(0, 1, 6)
ax.hist(per_q, bins=bins, color='seagreen', edgecolor='white')
ax.set_xlabel('Recall@100 of Flat vs supplied GT')
ax.set_ylabel('# queries')
ax.set_title(f'Per-query GT match — mean {per_q.mean():.4f} · std {per_q.std():.4f} · min {per_q.min():.2f}')
if per_q.max() <= per_q.min() + 1e-6:
    ax.text(0.5, 0.5, 'All queries identical recall\n(full-base GT match)', transform=ax.transAxes,
            ha='center', va='center', fontsize=10)
plt.tight_layout(); plt.savefig(DOCS_IMG / '01_gt_recall_hist.png', dpi=120, bbox_inches='tight'); plt.show()

if int(os.environ.get('LAB_LIGHT', '0')):
    print('LAB_LIGHT: skipping strict GT assert (subset index vs full-base GT is not comparable).')
else:
    assert per_q.mean() > 0.95, f"Flat-vs-GT recall too low ({per_q.mean():.3f}); dataset suspect"
del flat; import gc; gc.collect()

## 7 · Persist HDF5 metadata

We only store **query + GT + metadata** in HDF5 — the base file stays as the original
`.fvecs` and is accessed via `np.memmap` from later notebooks. Duplicating 10 GB of base
features into HDF5 would waste disk and yields no benefit because raw `.fvecs` is already
contiguous float32.

In [ ]:
with h5py.File(PATHS['h5'], 'w') as h:
    h.attrs['source']    = 'imagenet1m (ZJU CAD lab)'
    h.attrs['dim']       = dim
    h.attrs['n_base']    = n_base
    h.attrs['n_query']   = n_query
    h.attrs['gt_k']      = k_gt
    h.attrs['base_path'] = str(PATHS['base'])  # relative — portable across hosts/containers
    h.create_dataset('query',       data=queries, compression='gzip', compression_opts=4)
    h.create_dataset('groundtruth', data=gt,      compression='gzip', compression_opts=4)
print(f"wrote {PATHS['h5']}  ({PATHS['h5'].stat().st_size/1e6:.1f} MB)")

with h5py.File(PATHS['h5'], 'r') as h:
    print({k: h.attrs[k] for k in h.attrs.keys()})
    print('datasets:', list(h.keys()))

## Summary

* Dataset downloaded and integrity verified.
* `IndexFlatL2` matches the supplied ground truth at Recall@100 ≈ 1.0, confirming the
  dataset files are intact and our `fvecs`/`ivecs` readers are correct.
* All exploration figures saved under `docs/img/light/` or `docs/img/full/` depending on `LAB_LIGHT`.
* `data/imagenet1m.h5` stores the query + GT; the base set is accessed via memmap from
  `data/imagenet_base.fvecs`.

→ Proceed to **`02_ivf_benchmark.ipynb`**.